# Train YOLO Monster Detector

This notebook mirrors `train/train_yolo_monster.py`, but keeps each step editable for debugging.

In [ ]:
from pathlib import Path
import shutil
import yaml

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "train":
    REPO_ROOT = REPO_ROOT.parent

IMAGE_EXTS = {".bmp", ".jpg", ".jpeg", ".png", ".webp"}
print(REPO_ROOT)

## Parameters

In [ ]:
dataset_dir = REPO_ROOT / "train/datasets/monster_yolo"
output_dir = REPO_ROOT / "train/models/monster_yolo"

class_name = "monster"
base_model = "yolo11n.pt"
imgsz = 416
epochs = 80
batch = 16
device = "cuda"  # Use "cpu", "cuda", "0", or "mps"
patience = 20
workers = 4
run_name = "train"
exist_ok = True
export_format = "onnx"  # Use "", "onnx", "engine", or "coreml"

## Dataset Check

In [ ]:
def count_files(path, suffixes=None):
    if not path.exists():
        return 0
    if suffixes is None:
        return sum(1 for item in path.iterdir() if item.is_file())
    return sum(1 for item in path.iterdir() if item.is_file() and item.suffix.lower() in suffixes)


required_dirs = [
    dataset_dir / "images" / "train",
    dataset_dir / "images" / "val",
    dataset_dir / "labels" / "train",
    dataset_dir / "labels" / "val",
]
missing = [path for path in required_dirs if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required dataset directories:\n" + "\n".join(f"  - {p}" for p in missing))

train_images = count_files(dataset_dir / "images" / "train", IMAGE_EXTS)
val_images = count_files(dataset_dir / "images" / "val", IMAGE_EXTS)
train_labels = count_files(dataset_dir / "labels" / "train", {".txt"})
val_labels = count_files(dataset_dir / "labels" / "val", {".txt"})

print("Dataset summary:")
print(f"  train images: {train_images}")
print(f"  train labels: {train_labels}")
print(f"  val images:   {val_images}")
print(f"  val labels:   {val_labels}")

if train_images == 0:
    raise RuntimeError(f"No training images found: {dataset_dir / 'images' / 'train'}")
if val_images == 0:
    raise RuntimeError(f"No validation images found: {dataset_dir / 'images' / 'val'}")
if train_labels < train_images:
    print("Warning: fewer train label files than images. Empty scenes should still have empty .txt files.")
if val_labels < val_images:
    print("Warning: fewer val label files than images. Empty scenes should still have empty .txt files.")

## Generate `data.yaml`

In [ ]:
data_yaml = dataset_dir / "data.yaml"
data = {
    "path": str(dataset_dir.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {0: class_name},
}

if data_yaml.exists():
    with data_yaml.open("r", encoding="utf-8") as f:
        existing = yaml.safe_load(f) or {}
    existing.setdefault("path", data["path"])
    existing.setdefault("train", data["train"])
    existing.setdefault("val", data["val"])
    existing.setdefault("names", data["names"])
    data = existing

with data_yaml.open("w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

print(data_yaml)
print(data)

## Train

In [ ]:
from ultralytics import YOLO

model = YOLO(base_model)
results = model.train(
    data=str(data_yaml),
    imgsz=imgsz,
    epochs=epochs,
    batch=batch,
    device=device,
    project=str(output_dir),
    name=run_name,
    patience=patience,
    workers=workers,
    exist_ok=exist_ok,
)

run_dir = Path(results.save_dir)
print(run_dir)

## Copy `best.pt`

In [ ]:
best_pt = run_dir / "weights" / "best.pt"
if not best_pt.exists():
    raise FileNotFoundError(f"best.pt not found: {best_pt}")

output_dir.mkdir(parents=True, exist_ok=True)
copied_best_pt = output_dir / "best.pt"
shutil.copy2(best_pt, copied_best_pt)
print(copied_best_pt)

## Optional Export

In [ ]:
if export_format:
    export_model = YOLO(str(copied_best_pt))
    exported = export_model.export(format=export_format, imgsz=imgsz, device=device)
    exported_path = Path(exported)
    if exported_path.exists():
        dest = output_dir / exported_path.name
        shutil.copy2(exported_path, dest)
        print(dest)
    else:
        print(exported)
else:
    print("Export skipped")